## **Upload Dataset**

In [1]:
from google.colab import files
uploaded=files.upload()

Saving bank.csv to bank.csv


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN,Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization

## **Reading the data**


In [62]:
data=pd.read_csv('bank.csv',sep=';')

### **Data Head**

In [63]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


## **Data Information**

In [64]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


## **Data Summary**

In [65]:
data.describe()

,age,balance,day,duration,campaign,pdays,previous
count,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000
mean,41.170095,1422.657819,15.915284,263.961292,2.793630,39.766645,0.542579
std,10.576211,3009.638142,8.247667,259.856633,3.109807,100.121124,1.693562
min,19.000000,-3313.000000,1.000000,4.000000,1.000000,-1.000000,0.000000
25%,33.000000,69.000000,9.000000,104.000000,1.000000,-1.000000,0.000000
50%,39.000000,444.000000,16.000000,185.000000,2.000000,-1.000000,0.000000
75%,49.000000,1480.000000,21.000000,329.000000,3.000000,-1.000000,0.000000
max,87.000000,71188.000000,31.000000,3025.000000,50.000000,871.000000,25.000000


## **Data Preprocessing**

### **Identifying and handling misisng values in dataset**

In [66]:
pd.isnull(data).sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


### **Identifying duplicates and removing it**

In [67]:
data.duplicated().sum()

0

# **Addressing Outliers**

# **Identify categorical columns**


In [68]:
print(f"Various Job Categories in dataset :{data['job'].unique()}")

print(f"Various Marital Status in dataset:{data['marital'].unique()}")

print(f"Various education Status in dataset:{data['education'].unique()}")

print(f"Various defalut Status in dataset:{data['default'].unique()}")

print(f"Various housing Status in dataset:{data['housing'].unique()}")

print(f"Various loan status in dataset:{data['loan'].unique()}")

print(f"Various contact status in dataset:{data['contact'].unique()}")

print(f"Various month status in dataset:{data['month'].unique()}")

print(f"Various poutcome status in dataset:{data['poutcome'].unique()}")

print(f"Various Y(Target variable ) status in dataset:{data['y'].unique()}")

Various Job Categories in dataset :['unemployed' 'services' 'management' 'blue-collar' 'self-employed'
 'technician' 'entrepreneur' 'admin.' 'student' 'housemaid' 'retired'
 'unknown']
Various Marital Status in dataset:['married' 'single' 'divorced']
Various education Status in dataset:['primary' 'secondary' 'tertiary' 'unknown']
Various defalut Status in dataset:['no' 'yes']
Various housing Status in dataset:['no' 'yes']
Various loan status in dataset:['no' 'yes']
Various contact status in dataset:['cellular' 'unknown' 'telephone']
Various month status in dataset:['oct' 'may' 'apr' 'jun' 'feb' 'aug' 'jan' 'jul' 'nov' 'sep' 'mar' 'dec']
Various poutcome status in dataset:['unknown' 'failure' 'other' 'success']
Various Y(Target variable ) status in dataset:['no' 'yes']


# **One Hot colding to convert categorical columns into numerical values for further implementations**

In [69]:
# Identify categorical columns
binary_cols = ['default', 'housing', 'loan', 'y']  # Example binary columns (yes/no)
multi_category_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']  # Multi-category

# Convert binary columns to 0 & 1
for col in binary_cols:
    data[col] = data[col].map({'yes':1, 'no':0})

# Apply one-hot encoding for multi-category columns
data = pd.get_dummies(data, columns=multi_category_cols, drop_first=True)

# Display transformed dataset
print(data.head())


   age  default  balance  housing  loan  day  duration  campaign  pdays  \
0   30        0     1787        0     0   19        79         1     -1   
1   33        0     4789        1     1   11       220         1    339   
2   35        0     1350        1     0   16       185         1    330   
3   30        0     1476        1     1    3       199         4     -1   
4   59        0        0        1     0    5       226         1     -1   

   previous  ...  month_jul  month_jun  month_mar  month_may  month_nov  \
0         0  ...      False      False      False      False      False   
1         4  ...      False      False      False       True      False   
2         1  ...      False      False      False      False      False   
3         0  ...      False       True      False      False      False   
4         0  ...      False      False      False       True      False   

   month_oct  month_sep  poutcome_other  poutcome_success  poutcome_unknown  
0       True      Fa

In [70]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 43 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   age                  4521 non-null   int64
 1   default              4521 non-null   int64
 2   balance              4521 non-null   int64
 3   housing              4521 non-null   int64
 4   loan                 4521 non-null   int64
 5   day                  4521 non-null   int64
 6   duration             4521 non-null   int64
 7   campaign             4521 non-null   int64
 8   pdays                4521 non-null   int64
 9   previous             4521 non-null   int64
 10  y                    4521 non-null   int64
 11  job_blue-collar      4521 non-null   bool 
 12  job_entrepreneur     4521 non-null   bool 
 13  job_housemaid        4521 non-null   bool 
 14  job_management       4521 non-null   bool 
 15  job_retired          4521 non-null   bool 
 16  job_self-employed    452

# **Feature Selection**

In [71]:
X=data.drop('y',axis=1)
y=data['y']

# **Spliting the dataset**

In [72]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

## **Scaling the Features**
### We will use StandardScaling for scaling the feature since we are using neural network (usually prefered)

In [73]:
Scaler=StandardScaler()
X_train_scaled=Scaler.fit_transform(X_train)
X_test_scaled=Scaler.fit_transform(X_test)

### **Building the Recurrent neural network Model**

In [74]:
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))


# Define RNN model
model = Sequential()

# Input layer and first RNN layer
model.add(SimpleRNN(32, input_shape=(1, X_train_scaled.shape[1]),kernel_regularizer=l2(0.01),activation='relu', return_sequences=True))
model.add(Dropout(0.3))

# Second RNN layer
model.add(SimpleRNN(12, activation='relu'))
model.add(Dropout(0.3))

# Output layer (1 neuron for binary classification)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_14 (SimpleRNN)            │ (None, 1, 32)               │           2,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 1, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_15 (SimpleRNN)            │ (None, 12)                  │             540 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 12)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 1)                   │              13 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,953 (11.54 KB)

 Trainable params: 2,953 (11.54 KB)

 Non-trainable params: 0 (0.00 B)

## **Train the model**

In [75]:
# Train the model
history = model.fit(X_train_rnn, y_train, epochs=50, batch_size=32, validation_data=(X_test_rnn, y_test))


Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6427 - loss: 0.9708 - val_accuracy: 0.8909 - val_loss: 0.6217
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8585 - loss: 0.6707 - val_accuracy: 0.8946 - val_loss: 0.5098
Epoch 3/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8854 - loss: 0.5144 - val_accuracy: 0.8946 - val_loss: 0.4365
Epoch 4/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8900 - loss: 0.4487 - val_accuracy: 0.8931 - val_loss: 0.3839
Epoch 5/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8863 - loss: 0.4085 - val_accuracy: 0.8976 - val_loss: 0.3478
Epoch 6/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9029 - loss: 0.3406 - val_accuracy: 0.8968 - val_loss: 0.3235
Epoch 7/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9024 - loss: 0.3277 - val_accuracy: 0.8954 - val_loss: 0.3061
Epoch 8/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9017 - loss: 0.3140 - val_accuracy: 0.8931 - val_loss

## **Prediction**

In [77]:
# Make predictions
y_pred_prob = model.predict(X_test_rnn)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


## **Prediction output**

In [78]:
for i, prediction in enumerate(y_pred):
  print(f"customer {i+1} will {'subscribe' if prediction == 1 else 'not subscribe'} to subscription  ")

customer 1 will not subscribe to subscription  
customer 2 will subscribe to subscription  
customer 3 will not subscribe to subscription  
customer 4 will not subscribe to subscription  
customer 5 will not subscribe to subscription  
customer 6 will not subscribe to subscription  
customer 7 will not subscribe to subscription  
customer 8 will not subscribe to subscription  
customer 9 will not subscribe to subscription  
customer 10 will not subscribe to subscription  
customer 11 will not subscribe to subscription  
customer 12 will not subscribe to subscription  
customer 13 will not subscribe to subscription  
customer 14 will subscribe to subscription  
customer 15 will not subscribe to subscription  
customer 16 will not subscribe to subscription  
customer 17 will not subscribe to subscription  
customer 18 will not subscribe to subscription  
customer 19 will not subscribe to subscription  
customer 20 will not subscribe to subscription  
customer 21 will not subscribe to sub

## **Count No. of Employee able or unable to subscribe**

In [79]:
count_able = 0
count_unable=0

for prediction in y_pred:
  if prediction ==1 :
    count_able +=1
  else:
    count_unable +=1

print(f'No. of customer able to subscribe {count_able}')

print(f'No. of customer not able to subscribe {count_unable}')

No. of customer able to subscribe 102
No. of customer not able to subscribe 1255


# **Evaluate Model**

In [80]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_rnn, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")



43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8963 - loss: 0.2630
Test Accuracy: 0.8983


## **Whether user will make more than 50k a year or not using single and multilayer neural**




## **Selecting Features**

In [100]:
# Define column names correctly (as lists of column names, not dataframes)
work_area_cols = ['job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management',
                  'job_retired', 'job_self-employed', 'job_services', 'job_student',
                  'job_technician', 'job_unemployed', 'job_unknown']

qualification_cols = ['education_secondary', 'education_tertiary', 'education_unknown']

marital_status_cols = ['marital_married', 'marital_single']

# Selecting relevant features from the dataframe
X = data[work_area_cols + qualification_cols + marital_status_cols]

# Target variable
y = data['y']

## **Spliting the dataset**

In [101]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

## **Scaling the Features**


In [102]:
Scaler=StandardScaler()
X_train_scaled=Scaler.fit_transform(X_train)
X_test_scaled=Scaler.fit_transform(X_test)

## **Building the Recurrent neural network Model**

In [103]:
# Reshape for RNN input (samples, time_steps, features)
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# Define the RNN model with a single neuron
model = Sequential([
    SimpleRNN(1, input_shape=(1, X_train_rnn.shape[2])),  # Correct input shape
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train_rnn, y_train, epochs=50, batch_size=16, validation_data=(X_test_rnn, y_test))



Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


198/198 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5071 - loss: 0.6999 - val_accuracy: 0.6640 - val_loss: 0.6165
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6878 - loss: 0.6025 - val_accuracy: 0.8165 - val_loss: 0.5371
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8148 - loss: 0.5288 - val_accuracy: 0.8615 - val_loss: 0.4723
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8633 - loss: 0.4533 - val_accuracy: 0.8644 - val_loss: 0.4221
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8565 - loss: 0.4130 - val_accuracy: 0.8644 - val_loss: 0.3913
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8570 - loss: 0.3896 - val_accuracy: 0.8644 - val_loss: 0.3753
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8580 - loss: 0.3778 - val_accuracy: 0.8644 - val_loss: 0.3672
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8449 - loss: 0.3875 - val_accuracy: 0.8644 - val_

## **Predictions**

In [104]:
# Make predictions
y_pred_prob = model.predict(X_test_rnn)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)


43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


### **Prediction Output**

In [105]:
for i, prediction in enumerate(y_pred):
  print(f"customer {i+1} will {'subscribe' if prediction == 1 else 'not subscribe'} to subscription  ")


customer 1 will not subscribe to subscription  
customer 2 will not subscribe to subscription  
customer 3 will not subscribe to subscription  
customer 4 will not subscribe to subscription  
customer 5 will not subscribe to subscription  
customer 6 will not subscribe to subscription  
customer 7 will not subscribe to subscription  
customer 8 will not subscribe to subscription  
customer 9 will not subscribe to subscription  
customer 10 will not subscribe to subscription  
customer 11 will not subscribe to subscription  
customer 12 will not subscribe to subscription  
customer 13 will not subscribe to subscription  
customer 14 will not subscribe to subscription  
customer 15 will not subscribe to subscription  
customer 16 will not subscribe to subscription  
customer 17 will not subscribe to subscription  
customer 18 will not subscribe to subscription  
customer 19 will not subscribe to subscription  
customer 20 will not subscribe to subscription  
customer 21 will not subscrib

## **Number of People will be able to make more than 50K a year**

In [106]:
count_able = 0
count_unable=0

for prediction in y_pred:
  if prediction ==1 :
    count_able +=1
  else:
    count_unable +=1

print(f'No. of customer able to subscribe {count_able}')

print(f'No. of customer not able to subscribe {count_unable}')


No. of customer able to subscribe 0
No. of customer not able to subscribe 1357


# **Evaluate  Model**

In [107]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_rnn, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")



43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8899 - loss: 0.3445
Test Accuracy: 0.8880


## **Multi layers neuron**

## **Selecting Features**

In [108]:
# Define column names correctly (as lists of column names, not dataframes)
work_area_cols = ['job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management',
                  'job_retired', 'job_self-employed', 'job_services', 'job_student',
                  'job_technician', 'job_unemployed', 'job_unknown']

qualification_cols = ['education_secondary', 'education_tertiary', 'education_unknown']

marital_status_cols = ['marital_married', 'marital_single']

# Selecting relevant features from the dataframe
X = data[work_area_cols + qualification_cols + marital_status_cols]

# Target variable
y = data['y']

# **Spliting Dataset**

In [109]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

# **Scaling Dataset**

In [110]:
Scaler=StandardScaler()
X_train_scaled=Scaler.fit_transform(X_train)
X_test_scaled=Scaler.fit_transform(X_test)

# **Building Neural Network**

In [111]:
# Reshape for RNN input (samples, time_steps, features)
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# Define the Multi-Layer RNN Model
model = Sequential([
    SimpleRNN(32, return_sequences=True, input_shape=(1, X_train_rnn.shape[2])),  # First RNN Layer
    SimpleRNN(16, return_sequences=False),  # Second RNN Layer
    Dense(4, activation='relu'),  # Fully Connected Hidden Layer
    Dense(1, activation='sigmoid')  # Output Layer for Binary Classification
])

# Compile the Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the Model
history = model.fit(X_train_rnn, y_train, epochs=50, batch_size=16, validation_data=(X_test_rnn, y_test))


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


198/198 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7651 - loss: 0.6116 - val_accuracy: 0.8880 - val_loss: 0.3800
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8887 - loss: 0.3548 - val_accuracy: 0.8880 - val_loss: 0.3530
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8709 - loss: 0.3699 - val_accuracy: 0.8880 - val_loss: 0.3501
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8790 - loss: 0.3610 - val_accuracy: 0.8880 - val_loss: 0.3502
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8891 - loss: 0.3390 - val_accuracy: 0.8873 - val_loss: 0.3501
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8787 - loss: 0.3585 - val_accuracy: 0.8865 - val_loss: 0.3505
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8779 - loss: 0.3562 - val_accuracy: 0.8865 - val_loss: 0.3498
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8839 - loss: 0.3442 - val_accuracy: 0.8865 - val_

## **Predictions**

In [112]:
# Make predictions
y_pred_prob = model.predict(X_test_rnn)

# Convert probabilities to binary labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


## **Prediction**

In [113]:
for i, prediction in enumerate(y_pred):
    print(f'Customer {i+1} will {"be able" if prediction == 1 else "not be able"} to make more than 50k a year.')

Customer 1 will not be able to make more than 50k a year.
Customer 2 will not be able to make more than 50k a year.
Customer 3 will not be able to make more than 50k a year.
Customer 4 will not be able to make more than 50k a year.
Customer 5 will not be able to make more than 50k a year.
Customer 6 will not be able to make more than 50k a year.
Customer 7 will not be able to make more than 50k a year.
Customer 8 will not be able to make more than 50k a year.
Customer 9 will not be able to make more than 50k a year.
Customer 10 will not be able to make more than 50k a year.
Customer 11 will not be able to make more than 50k a year.
Customer 12 will not be able to make more than 50k a year.
Customer 13 will not be able to make more than 50k a year.
Customer 14 will be able to make more than 50k a year.
Customer 15 will not be able to make more than 50k a year.
Customer 16 will not be able to make more than 50k a year.
Customer 17 will not be able to make more than 50k a year.
Customer 1

#**Number of People will be able to make more than 50K a year**

In [114]:
count_able = 0
count_unable=0

for prediction in y_pred:
  if prediction ==1 :
    count_able +=1
  else:
    count_unable +=1

print(f'No. of customer able to subscribe {count_able}')

print(f'No. of customer not able to subscribe {count_unable}')


No. of customer able to subscribe 13
No. of customer not able to subscribe 1344


# **Evaluate  Model**

In [115]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_rnn, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")


43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8838 - loss: 0.3543
Test Accuracy: 0.8843
